### Carga de Dependencias

In [1]:
import pandas as pd

### Carga de datos y variables

In [2]:
df_train = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_train.parquet", engine="pyarrow")
df_test = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_test.parquet", engine="pyarrow")
df_oot = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_oot.parquet", engine="pyarrow")
df_oos = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_oos.parquet", engine="pyarrow")

In [3]:
varss = ['attribute4_change_rate',
 'attribute2',
 'attribute8',
 'attribute1_lag_7_max',
 'attribute1_lag_120_max',
 'attribute1_lag_120_mean',
 'attribute5_lag_15_mean',
 'attribute5_lag_60_mean',
 'attribute6_lag_15_std',
 'attribute6_lag_30_min',
 'attribute6_lag_30_std',
 'attribute6_lag_60_min',
 'attribute6_lag_90_std',
 'attribute2_change_rate',
 'attribute2_cumulative_change',
 'attribute4_cumulative_change']

### División de muestras

In [4]:
x_train = df_train[varss].copy()
y_train = df_train['failure'].copy()

x_test = df_test[varss]
y_test = df_test['failure']

x_oot = df_oot[varss].copy()
y_oot = df_oot['failure'].copy()

x_oos = df_oos[varss].copy()
y_oos = df_oos['failure'].copy()
    

### Optimización hiperparámetros

In [5]:
!pip install optuna

  Using cached colorlog-6.9.0-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/400.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/400.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/400.9 kB ? eta -:--:--
   --- ----------------------------------- 41.0/400.9 kB 279.3 kB/s eta 0:00:02
   ---------- --------------------------- 112.6/400.9 kB 652.2 kB/s eta 0:00:01
   ----------- -------------------------- 122.9/400.9 kB 654.9 kB/s eta 0:00:01
   ----------------------- ---------------- 235.5/400.9 kB 1.0 MB/s eta 0:00:01
   ---------------------------------------  399.4/400.9 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 400.9/400.9 kB 1.3 MB/s eta 0:00:00
Using cached colorlog-6.9.0-py3-none-any.whl (11 kB)



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

# Calcular scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'verbose': -1
    }
    
    model = LGBMClassifier(**params)
    cv_scores = cross_val_score(model, x_train, y_train, cv=5, scoring='recall', n_jobs=-1)
    
    return cv_scores.mean()

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f"Mejor Recall (CV): {study_lgbm.best_value:.4f}")
print(f"Mejores parámetros:\n{study_lgbm.best_params}")

[I 2025-10-06 14:02:44,989] A new study created in memory with name: no-name-0b9c79f1-d570-44f4-859c-86cbf9242639


Scale pos weight: 977.51


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-10-06 14:03:03,629] Trial 0 finished with value: 0.5801470588235293 and parameters: {'n_estimators': 240, 'max_depth': 8, 'learning_rate': 0.21295228726151563, 'num_leaves': 53, 'min_child_samples': 93}. Best is trial 0 with value: 0.5801470588235293.
[I 2025-10-06 14:03:12,827] Trial 1 finished with value: 0.6588235294117647 and parameters: {'n_estimators': 285, 'max_depth': 5, 'learning_rate': 0.266149597737851, 'num_leaves': 74, 'min_child_samples': 94}. Best is trial 1 with value: 0.6588235294117647.
[I 2025-10-06 14:03:15,556] Trial 2 finished with value: 0.4477941176470589 and parameters: {'n_estimators': 175, 'max_depth': 9, 'learning_rate': 0.12280533116868564, 'num_leaves': 90, 'min_child_samples': 65}. Best is trial 1 with value: 0.6588235294117647.
[I 2025-10-06 14:03:19,475] Trial 3 finished with value: 0.5088235294117648 and parameters: {'n_estimators': 234, 'max_depth': 7, 'learning_rate': 0.05987696017279201, 'num_leaves': 69, 'min_child_samples': 96}. Best is tr

In [8]:
from catboost import CatBoostClassifier

# Calcular scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

def objective_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 500),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'verbose': False
    }
    
    model = CatBoostClassifier(**params)
    cv_scores = cross_val_score(model, x_train, y_train, cv=5, scoring='recall', n_jobs=-1)
    
    return cv_scores.mean()

study_catboost = optuna.create_study(direction='maximize')
study_catboost.optimize(objective_catboost, n_trials=50, show_progress_bar=True)

print(f"Mejor Recall (CV): {study_catboost.best_value:.4f}")
print(f"Mejores parámetros:\n{study_catboost.best_params}")

[I 2025-10-06 14:08:11,619] A new study created in memory with name: no-name-c8252f0e-1bc3-4db6-9217-b48dc0547db4


Scale pos weight: 977.51


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-10-06 14:08:39,892] Trial 0 finished with value: 0.65 and parameters: {'iterations': 297, 'depth': 4, 'learning_rate': 0.025621013648866077, 'l2_leaf_reg': 9.828992266874057}. Best is trial 0 with value: 0.65.
[I 2025-10-06 14:10:07,687] Trial 1 finished with value: 0.011764705882352941 and parameters: {'iterations': 250, 'depth': 10, 'learning_rate': 0.07206515793369705, 'l2_leaf_reg': 2.6941537236102}. Best is trial 0 with value: 0.65.
[I 2025-10-06 14:10:20,402] Trial 2 finished with value: 0.5036764705882353 and parameters: {'iterations': 199, 'depth': 6, 'learning_rate': 0.06042511658812677, 'l2_leaf_reg': 4.885984184495799}. Best is trial 0 with value: 0.65.
[I 2025-10-06 14:10:50,092] Trial 3 finished with value: 0.49191176470588244 and parameters: {'iterations': 289, 'depth': 5, 'learning_rate': 0.03112429629242045, 'l2_leaf_reg': 9.81747513817623}. Best is trial 0 with value: 0.65.
[I 2025-10-06 14:11:30,179] Trial 4 finished with value: 0.09558823529411765 and paramet

In [11]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
import numpy as np

def objective_histgb(trial):
    params = {
        'max_iter': trial.suggest_int('max_iter', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 100),
        'random_state': 42
    }
    
    model = HistGradientBoostingClassifier(**params)
    
    # CV manual para pasar sample_weight
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    recall_scores = []
    
    for train_idx, val_idx in cv.split(x_train, y_train):
        X_fold_train, X_fold_val = x_train.iloc[train_idx], x_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Calcular sample_weight para el fold
        sample_weights = np.ones(len(y_fold_train))
        sample_weights[y_fold_train == 1] = (y_fold_train == 0).sum() / (y_fold_train == 1).sum()
        
        model.fit(X_fold_train, y_fold_train, sample_weight=sample_weights)
        y_pred = model.predict(X_fold_val)
        
        # Calcular recall manualmente
        from sklearn.metrics import recall_score
        recall = recall_score(y_fold_val, y_pred)
        recall_scores.append(recall)
    
    return np.mean(recall_scores)

study_histgb = optuna.create_study(direction='maximize')
study_histgb.optimize(objective_histgb, n_trials=50, show_progress_bar=True)

print(f"Mejor Recall (CV): {study_histgb.best_value:.4f}")
print(f"Mejores parámetros:\n{study_histgb.best_params}")

[I 2025-10-06 20:06:35,957] A new study created in memory with name: no-name-348dae3f-2b18-4872-9363-3f07b406c845


  0%|          | 0/50 [00:00<?, ?it/s]

c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


[I 2025-10-06 20:06:43,999] Trial 0 finished with value: 0.3397058823529412 and parameters: {'max_iter': 302, 'max_depth': 6, 'learning_rate': 0.2701709287492109, 'min_samples_leaf': 38}. Best is trial 0 with value: 0.3397058823529412.


c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


...

[I 2025-10-06 20:08:15,104] Trial 47 finished with value: 0.6279411764705882 and parameters: {'max_iter': 371, 'max_depth': 9, 'learning_rate': 0.02099734286222737, 'min_samples_leaf': 97}. Best is trial 45 with value: 0.7227941176470588.


c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


[I 2025-10-06 20:08:16,675] Trial 48 finished with value: 0.6514705882352941 and parameters: {'max_iter': 274, 'max_depth': 4, 'learning_rate': 0.049346652873709096, 'min_samples_leaf': 100}. Best is trial 45 with value: 0.7227941176470588.


c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


[I 2025-10-06 20:08:17,790] Trial 49 finished with value: 0.6742647058823529 and parameters: {'max_iter': 351, 'max_depth': 3, 'learning_rate': 0.09947284003310251, 'min_samples_leaf': 38}. Best is trial 45 with value: 0.7227941176470588.
Mejor Recall (CV): 0.7228
Mejores parámetros:
{'max_iter': 314, 'max_depth': 3, 'learning_rate': 0.01931719708124155, 'min_samples_leaf': 90}
